In [7]:
# %% VSCode Jupyter: 경로만 수정해서 한 셀로 실행
from pathlib import Path
import pandas as pd
import re
import unicodedata as ud

# ========= 경로 설정 (여기만 수정) =========
INPUT_DIR  = Path(r"/Users/doyoung-gil/연구실/데이터/배/지역")              # 8개 지역 CSV 폴더 (파일명=지역명.csv)
COORDS_CSV = Path(r"/Users/doyoung-gil/연구실/데이터/배/경위도 데이터.csv") # '시도, 지점명, 경도, 위도' 열 포함
OUTPUT_DIR = Path(r"/Users/doyoung-gil/연구실/데이터/배/지역1")             # 결과 저장 폴더

# ========= 옵션 =========
ROUND_COORDS = None   # 예: 6 으로 설정하면 위경도 소수 6자리 반올림
ENCODINGS = ("utf-8-sig", "utf-8", "cp949", "euc-kr")

# 시/도 축약. (필요 시 더 추가 가능)
PROV_SHORT = {
    "경기도":"경기",
    "충청남도":"충남", "충청북도":"충북",
    "전라북도":"전북", "전라남도":"전남",
    "경상북도":"경북", "경상남도":"경남",
    "울산광역시":"울산", "부산광역시":"부산", "대구광역시":"대구",
    "광주광역시":"광주", "대전광역시":"대전", "세종특별자치시":"세종",
    "서울특별시":"서울", "인천광역시":"인천",
}
# (선택) 추가로 '충청'처럼 광역 축약도 허용하고 싶으면 여기 적어둔다.
PROV_ALT = {
    "충청남도":"충청", "충청북도":"충청",
}

# 특정 행(시도, 지점명)에 대해 원하는 별칭 키를 추가로 허용
SPECIAL_ALIASES = {
    # 전라남도 나주시 금천면 -> 기본키 "전남나주" 외에 "전남금천"도 허용
    ("전라남도", "나주시 금천면"): ["전남금천"],
    # 필요 시 추가:
    # ("충청남도", "천안시 입장면"): ["충청천안"],   # 기본키는 "충남천안"이지만 '충청천안'도 허용하고 싶다면
}

# ========= 유틸 =========
def nfc(x: str) -> str:
    return ud.normalize("NFC", str(x))

def read_csv_any_encoding(p: Path) -> pd.DataFrame:
    last_err = None
    for enc in ENCODINGS:
        try:
            return pd.read_csv(p, encoding=enc)
        except Exception as e:
            last_err = e
    try:
        return pd.read_csv(p)
    except Exception:
        raise last_err or RuntimeError(f"Failed to read CSV: {p}")

def normalize_region(s: str) -> str:
    # 파일명(확장자 제거)도 NFC로 통일
    return nfc(Path(str(s)).stem.strip())

def to_default_key(province: str, place: str) -> str:
    # '시도'와 '지점명'을 NFC로 통일하고, 지점명의 첫 토큰에서 '시/군/구'를 제거
    province = nfc(province)
    place    = nfc(place)
    prov_short = PROV_SHORT.get(province, province)
    first_token = place.split()[0] if place.strip() else ""
    first_token = re.sub(r"(시|군|구)$", "", first_token)
    return nfc(f"{prov_short}{first_token}")

def to_alt_key(province: str, place: str) -> str | None:
    # '충청남도' -> '충청' 같은 대체 접두를 쓰는 보조 키
    province = nfc(province)
    place    = nfc(place)
    alt = PROV_ALT.get(province)
    if not alt:
        return None
    first_token = place.split()[0] if place.strip() else ""
    first_token = re.sub(r"(시|군|구)$", "", first_token)
    return nfc(f"{alt}{first_token}")

# ========= 메인 배치 =========
def run_batch(input_dir: Path, coords_csv: Path, output_dir: Path):
    if not input_dir.exists():
        raise FileNotFoundError(f"Input dir not found: {input_dir}")
    if not coords_csv.exists():
        raise FileNotFoundError(f"Coordinate CSV not found: {coords_csv}")
    output_dir.mkdir(parents=True, exist_ok=True)

    coord_df = read_csv_any_encoding(coords_csv)

    # 좌표 파일의 열명 매핑 (기대: 시도, 지점명, 위도, 경도)
    colmap = {}
    for want, cands in {
        "시도":   ["시도","광역시도","province","시_도"],
        "지점명": ["지점명","지점","지역명","장소","name","station"],
        "위도":   ["위도","좌표-위도","lat","Latitude","LAT"],
        "경도":   ["경도","좌표-경도","lon","Longitude","LON"],
    }.items():
        for c in cands:
            if c in coord_df.columns:
                colmap[want] = c
                break
        if want not in colmap:
            raise KeyError(f"'{want}' 열을 좌표 파일에서 찾지 못했습니다. 현재 열들: {list(coord_df.columns)}")

    # NFC 정규화
    coord_df[colmap["시도"]]   = coord_df[colmap["시도"]].map(nfc)
    coord_df[colmap["지점명"]] = coord_df[colmap["지점명"]].map(nfc)

    # (선택) 좌표 반올림
    if ROUND_COORDS is not None:
        coord_df[colmap["위도"]] = pd.to_numeric(coord_df[colmap["위도"]], errors="coerce").round(ROUND_COORDS)
        coord_df[colmap["경도"]] = pd.to_numeric(coord_df[colmap["경도"]], errors="coerce").round(ROUND_COORDS)

    # 키 -> (lat, lon) 매핑 구축: 기본키 + 대체키 + 특수 별칭
    name_to_coord: dict[str, tuple[float, float]] = {}
    for _, row in coord_df.iterrows():
        prov  = row[colmap["시도"]]
        place = row[colmap["지점명"]]
        lat   = row[colmap["위도"]]
        lon   = row[colmap["경도"]]

        keys = {to_default_key(prov, place)}
        alt = to_alt_key(prov, place)
        if alt:
            keys.add(alt)
        # SPECIAL_ALIASES (NFC로 통일)
        for al in [nfc(x) for x in SPECIAL_ALIASES.get((prov, place), [])]:
            keys.add(al)

        for k in keys:
            if k:
                name_to_coord[k] = (lat, lon)

    print(f"[INFO] Built {len(name_to_coord)} keys from coords. Examples: {list(name_to_coord)[:10]}")

    # 입력 CSV 처리
    files = sorted([p for p in input_dir.glob("*.csv") if p.name != coords_csv.name])
    if not files:
        print(f"[WARN] No CSVs found in {input_dir}")
        return

    ok, miss = 0, 0
    for f in files:
        region_name = normalize_region(f.name)     # 파일명도 NFC
        if region_name not in name_to_coord:
            print(f"[WARN] No coordinates for '{region_name}' (filename: {f.name}). Skipped.")
            miss += 1
            continue

        lat, lon = name_to_coord[region_name]
        df = read_csv_any_encoding(f)

        prefix = pd.DataFrame({
            "지역명":    [region_name] * len(df),
            "좌표-위도": [lat]         * len(df),
            "좌표-경도": [lon]         * len(df),
        })

        out = pd.concat([prefix, df.reset_index(drop=True)], axis=1)
        out_path = output_dir / f.name
        out.to_csv(out_path, index=False, encoding="utf-8-sig")
        ok += 1
        print(f"[OK] Wrote: {out_path.name} (rows={len(out)})")

    print(f"[DONE] Success: {ok}, Skipped (no coords): {miss}, Total discovered: {len(files)}")
    print(f"[OUT]  Output directory: {output_dir}")

# ========= 실행 =========
run_batch(INPUT_DIR, COORDS_CSV, OUTPUT_DIR)


[INFO] Built 10 keys from coords. Examples: ['경기이천', '충남천안', '충청천안', '전북완주', '전남나주', '전남금천', '경북상주', '경북영천', '경남사천', '울산울주']
[OK] Wrote: 경기이천.csv (rows=1911)
[OK] Wrote: 경남사천.csv (rows=1911)
[OK] Wrote: 경북상주.csv (rows=1907)
[OK] Wrote: 경북영천.csv (rows=1911)
[OK] Wrote: 울산울주.csv (rows=1911)
[OK] Wrote: 전남나주.csv (rows=1911)
[OK] Wrote: 전북완주.csv (rows=1819)
[OK] Wrote: 충남천안.csv (rows=1910)
[DONE] Success: 8, Skipped (no coords): 0, Total discovered: 8
[OUT]  Output directory: /Users/doyoung-gil/연구실/데이터/배/지역1


In [ ]:
# make_moving_windows_for_fuzzy_pear.py
from pathlib import Path
import pandas as pd
import numpy as np

# ===== 경로/설정 =====
IN_DIR  = Path("/Users/doyoung-gil/연구실/데이터/배/site")       # 입력 CSV 폴더
OUT_DIR = IN_DIR / "1WeatherMovingAverage"                      # 출력 폴더
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR_START = 2017
YEAR_END   = 2023
YEARS = list(range(YEAR_START, YEAR_END + 1))

# ---- window 세팅 ----
OFFS = list(range(0, 30))   # offset 0~29OFFS  = [0, 5, 10, 15, 20, 25]
WIN  = 30                   # 창 길이(일)
STEP = 30                   # 창 간격(일) -> 12개면 360일 커버
MIN_DAYS = 26               # ✅ 결측 허용 기준: 26일 이상이면 window 통과

# 원본 열 이름 (배 데이터)
COL_DATE  = "일자"
COL_TMAX  = "최고 기온"
COL_TMIN  = "최저 기온"
COL_TAVG  = "평균 기온"
COL_WMEAN = "평균 풍속(m/s)"
COL_WGUST = "최대 풍속(m/s)"
COL_PRCP  = "강우량(mm)"

def parse_site_id(fname: str) -> str:
    return Path(fname).stem

def read_csv_any_encoding(p: Path) -> pd.DataFrame:
    for enc in ("utf-8-sig", "utf-8", "cp949", "euc-kr"):
        try:
            return pd.read_csv(p, encoding=enc, low_memory=False)
        except Exception:
            continue
    return pd.read_csv(p, low_memory=False)

def to_num(s: pd.Series) -> pd.Series:
    if s is None:
        return pd.Series(dtype="float64")
    # comma 제거/공백 제거 후 numeric
    ss = s.astype(str).str.replace(",", "", regex=False).str.strip()
    ss = ss.replace({"": np.nan, "nan": np.nan, "NaN": np.nan})
    return pd.to_numeric(ss, errors="coerce")

def window_stats(d: pd.DataFrame) -> dict:
    """30일 창 집계: 평균/최댓값/합계 + 관측일수(고유 날짜 수)."""
    out = {}

    tmax = to_num(d.get(COL_TMAX))
    tmin = to_num(d.get(COL_TMIN))
    tavg = to_num(d.get(COL_TAVG))

    # tavg 없으면 (tmax+tmin)/2로 보정(둘 다 있을 때만)
    if tavg is None or tavg.empty:
        tavg = pd.Series(index=d.index, dtype="float64")
    else:
        tavg = tavg.copy()
    fill_mask = (tmax.notna() & tmin.notna() & tavg.isna())
    tavg[fill_mask] = (tmax[fill_mask] + tmin[fill_mask]) / 2.0

    out["tavg_mean_30d"] = float(tavg.mean(skipna=True))
    out["tmin_mean_30d"] = float(tmin.mean(skipna=True))
    out["tmax_mean_30d"] = float(tmax.mean(skipna=True))

    wmean = to_num(d.get(COL_WMEAN))
    wgust = to_num(d.get(COL_WGUST))
    prcp  = to_num(d.get(COL_PRCP))

    out["wind_mean_30d"]     = float(wmean.mean(skipna=True))
    out["wind_gust_max_30d"] = float(wgust.max(skipna=True))
    out["precip_sum_30d"]    = float(prcp.sum(skipna=True))

    # ✅ 관측일수: 고유 날짜 수
    out["n_days_present"] = int(d[COL_DATE].nunique(dropna=True))
    return out

def gen_windows_for_year(year: int, offset_days: int):
    """
    시작연도 year의 1/1에서 offset_days만큼 이동한 날짜를 window1 시작으로 하고,
    STEP=30 간격으로 12개 window 생성. (연도 경계 넘어가도 OK)
    """
    base = pd.Timestamp(f"{year}-01-01")
    wins = []
    for i in range(12):
        start = base + pd.Timedelta(days=offset_days + i * STEP)
        end   = start + pd.Timedelta(days=WIN - 1)  # 포함 끝(30일)
        wins.append((i + 1, start, end))
    return wins

def process_one(fp: Path):
    df = read_csv_any_encoding(fp)
    if COL_DATE not in df.columns:
        print(f"[스킵] 날짜 컬럼 없음: {fp.name}")
        return

    # 날짜 파싱/정렬
    df[COL_DATE] = pd.to_datetime(df[COL_DATE], errors="coerce").dt.normalize()
    nat_cnt = int(df[COL_DATE].isna().sum())
    if nat_cnt > 0:
        # 날짜 파싱 실패행이 있으면 제거(기상값은 남아도 날짜가 없으면 window에 못 들어감)
        df = df.dropna(subset=[COL_DATE])

    df = df.sort_values(COL_DATE).reset_index(drop=True)

    if df.empty:
        print(f"[스킵] 유효 날짜 없음: {fp.name}")
        return

    data_min = df[COL_DATE].min()
    data_max = df[COL_DATE].max()

    site_id = parse_site_id(fp.name)
    rows = []

    for yr in YEARS:
        for o in OFFS:
            for idx, start, end in gen_windows_for_year(yr, o):

                # 창이 데이터 범위와 아예 안 겹치면 스킵
                if end < data_min or start > data_max:
                    continue

                win = df[(df[COL_DATE] >= start) & (df[COL_DATE] <= end)]
                if win.empty:
                    continue

                stats = window_stats(win)

                # ✅ 최소 관측일수 기준으로만 필터
                if stats["n_days_present"] < MIN_DAYS:
                    continue

                rows.append({
                    "site_id": site_id,
                    "year": yr,
                    "offset_days": o,
                    "window_idx": idx,
                    "start_date": start.date(),
                    "end_date": end.date(),
                    "days": WIN,
                    **stats
                })

    if not rows:
        print(f"[정보] 통과 window 없음: {fp.name}")
        return

    out = pd.DataFrame(rows)

    # 컬럼 순서
    COL_ORDER = [
        "site_id","year","offset_days","window_idx",
        "start_date","end_date","days",
        "tavg_mean_30d","tmin_mean_30d","tmax_mean_30d",
        "wind_mean_30d","wind_gust_max_30d","precip_sum_30d",
        "n_days_present",
    ]
    out = out[COL_ORDER].sort_values(["year","offset_days","window_idx"]).reset_index(drop=True)

    out_fname = f"{site_id}_WeatherMovingAverage.csv"
    out_path  = OUT_DIR / out_fname
    out.to_csv(out_path, index=False, encoding="utf-8-sig", float_format="%.3f")

    # 요약 로그
    print(f"[저장] {out_path} rows={len(out):,} (MIN_DAYS={MIN_DAYS}) "
          f"data_range={data_min.date()}~{data_max.date()}")

def main():
    files = sorted(IN_DIR.glob("*.csv"))
    if not files:
        print(f"[종료] 입력 없음: {IN_DIR}")
        return
    for fp in files:
        process_one(fp)

if __name__ == "__main__":
    main()


[저장] /Users/doyoung-gil/연구실/데이터/배/site/1WeatherMovingAverage/경기이천_WeatherMovingAverage.csv rows=1,859 (MIN_DAYS=26) data_range=2017-12-12~2023-03-06
[저장] /Users/doyoung-gil/연구실/데이터/배/site/1WeatherMovingAverage/경남사천_WeatherMovingAverage.csv rows=1,859 (MIN_DAYS=26) data_range=2017-12-12~2023-03-06
[저장] /Users/doyoung-gil/연구실/데이터/배/site/1WeatherMovingAverage/경북상주_WeatherMovingAverage.csv rows=1,859 (MIN_DAYS=26) data_range=2017-12-12~2023-03-06
[저장] /Users/doyoung-gil/연구실/데이터/배/site/1WeatherMovingAverage/경북영천_WeatherMovingAverage.csv rows=1,859 (MIN_DAYS=26) data_range=2017-12-12~2023-03-06
[저장] /Users/doyoung-gil/연구실/데이터/배/site/1WeatherMovingAverage/울산울주_WeatherMovingAverage.csv rows=1,859 (MIN_DAYS=26) data_range=2017-12-12~2023-03-06
[저장] /Users/doyoung-gil/연구실/데이터/배/site/1WeatherMovingAverage/전남나주_WeatherMovingAverage.csv rows=1,859 (MIN_DAYS=26) data_range=2017-12-12~2023-03-06
[저장] /Users/doyoung-gil/연구실/데이터/배/site/1WeatherMovingAverage/전ᄇ

/var/folders/2j/__kxcrvx5ss949vx7sspc6th0000gn/T/ipykernel_73154/3126778675.py:46: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ss = ss.replace({"": np.nan, "nan": np.nan, "NaN": np.nan})
/var/folders/2j/__kxcrvx5ss949vx7sspc6th0000gn/T/ipykernel_73154/3126778675.py:46: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ss = ss.replace({"": np.nan, "nan": np.nan, "NaN": np.nan})
/var/folders/2j/__kxcrvx5ss949vx7sspc6th0000gn/T/ipykernel_73154/3126778675.py:46: FutureWarning: Downcasting behavior in `replace` is deprecated and will be 

[저장] /Users/doyoung-gil/연구실/데이터/배/site/1WeatherMovingAverage/충남천안_WeatherMovingAverage.csv rows=1,859 (MIN_DAYS=26) data_range=2017-12-12~2023-03-06


In [19]:
from pathlib import Path
import pandas as pd
import unicodedata
import re

BEST_DIR    = Path("/Users/doyoung-gil/연구실/데이터/배/site/1Fuzzy/best")
REGION_DIR  = Path("/Users/doyoung-gil/연구실/데이터/배/site")
OUT_DIR     = BEST_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

def read_csv_any(p: Path) -> pd.DataFrame:
    for enc in ("utf-8-sig", "utf-8", "cp949", "euc-kr"):
        try:
            return pd.read_csv(p, encoding=enc, low_memory=False)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(p, low_memory=False)

def norm_key(x):
    s = "" if pd.isna(x) else str(x)
    # 유니코드 정규화(NFC) + 제로폭/제어문자 제거 + 모든 공백 제거 + 좌우 strip
    s = unicodedata.normalize("NFC", s)
    s = re.sub(r"[\u200B-\u200D\uFEFF]", "", s)   # zero-width
    s = s.strip().replace(" ", "")
    return s

# 1) 지역 CSV에서 매핑 만들기
region_files = sorted(REGION_DIR.glob("*.csv"))
if not region_files:
    raise FileNotFoundError(f"지역 CSV 없음: {REGION_DIR}")

maps = []
for fp in region_files:
    df = read_csv_any(fp)
    need = ["지역명", "좌표-위도", "좌표-경도"]
    if not set(need).issubset(df.columns):
        raise ValueError(f"[ERROR] 필요한 컬럼 없음: {fp.name} / 필요: {need}")
    df = df[need].drop_duplicates(subset=["지역명"])
    df["__key__"] = df["지역명"].apply(norm_key)
    maps.append(df[["__key__","좌표-위도","좌표-경도"]])

coord_map = pd.concat(maps, ignore_index=True).drop_duplicates("__key__", keep="last")

# 2) best_*에 좌표 붙이기
best_files = sorted(BEST_DIR.glob("best_*.csv"))
if not best_files:
    raise FileNotFoundError(f"best 파일 없음: {BEST_DIR}")

for fp in best_files:
    best = read_csv_any(fp)
    if "site_id" not in best.columns:
        print(f"[WARN] site_id 없음: {fp.name} → 스킵")
        continue

    best["__key__"] = best["site_id"].apply(norm_key)
    merged = best.merge(coord_map, on="__key__", how="left").drop(columns=["__key__"])

    # 칼럼 재배치: site_id 오른쪽에 좌표 2개
    cols = list(merged.columns)
    if "좌표-위도" in cols and "좌표-경도" in cols:
        i = cols.index("site_id")
        for c in ["좌표-위도","좌표-경도"]:
            cols.pop(cols.index(c))
        cols = cols[:i+1] + ["좌표-위도","좌표-경도"] + cols[i+1:]
        merged = merged[cols]

    # 미매칭 진단
    miss = merged["site_id"][merged["좌표-위도"].isna() | merged["좌표-경도"].isna()].unique()
    if len(miss) > 0:
        print(f"[WARN] 좌표 미매칭 ({fp.name}): {', '.join(map(str, miss))}")

    out_path = OUT_DIR / fp.name
    merged.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"[저장] {out_path}  rows={len(merged)}")


KeyError: '좌표-위도'

In [21]:
# merge_best_and_dates_to_doy.py
from pathlib import Path
import pandas as pd

# 경로 설정 
BEST_ROOT = Path("/Users/doyoung-gil/연구실/데이터/배/site/1Fuzzy/best")
OUT_CSV   = BEST_ROOT / "best_all_sites_DOY.csv"

# DOY로 변환할 날짜 컬럼들 (파일에 있는 그대로 이름 사용)
DATE_COLS = [
    "dormancy_start_date",
    "dormancy_end_date",
    "growing_start_date",
    "growing_end_date",
    "flowering_date",
]

# 날짜 -> DOY (1~366, 결측은 NA) 변환
def to_doy(series: pd.Series) -> pd.Series:
    dt = pd.to_datetime(series, errors="coerce", format="%Y-%m-%d")
    return dt.dt.dayofyear.astype("Int64")  # pandas nullable integer

# === 파일 수집 ===
files = sorted(set(BEST_ROOT.rglob("best_*.csv")) | set(BEST_ROOT.rglob("BEST_*.csv")))
print(f"[발견] 대상 파일: {len(files)}개")

# === 읽기 & 합치기 ===
dfs = []
for f in files:
    try:
        df = pd.read_csv(f, encoding="utf-8-sig")
    except UnicodeDecodeError:
        df = pd.read_csv(f, encoding="utf-8")

    # 날짜 컬럼을 DOY로 교체 (존재하는 컬럼만 처리)
    for col in DATE_COLS:
        if col in df.columns:
            df[col] = to_doy(df[col])

    # site_id 바로 뒤에 좌표 배치(이미 있다면 순서만 정리)
    need = ["site_id", "좌표-위도", "좌표-경도"]
    if all(c in df.columns for c in need):
        cols = list(df.columns)
        # 좌표 두 컬럼을 현 위치에서 제거 후 site_id 뒤에 재삽입
        cols.remove("좌표-위도"); cols.remove("좌표-경도")
        pos = cols.index("site_id") + 1
        cols = cols[:pos] + ["좌표-위도", "좌표-경도"] + cols[pos:]
        df = df[cols]

    dfs.append(df)

if not dfs:
    raise SystemExit("[중단] 병합할 파일이 없습니다.")

merged = pd.concat(dfs, ignore_index=True)

# === 저장 ===
merged.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print(f"[저장] {OUT_CSV}  rows={len(merged)}  cols={len(merged.columns)}")
print("[완료] 모든 날짜 컬럼을 DOY로 변환하여 합본 저장.")

[발견] 대상 파일: 8개
[저장] /Users/doyoung-gil/연구실/데이터/배/site/1Fuzzy/best/best_all_sites_DOY.csv  rows=40  cols=14
[완료] 모든 날짜 컬럼을 DOY로 변환하여 합본 저장.
